# Phase 1 - Data Processing

Predict hourly departures per station so ops can rebalance before stations run out.

**In:** `data/export` (warehouse export, 18.9M trips, 2019-2023)  
**Out:** `data/cleaned` - filtered fact table, plus `station_pool.parquet`

| Step | What |
| --- | --- |
| 1 | Load |
| 2 | Profile and validate |
| 3 | Station pool: top 50 in any year, operating the full span |
| 4 | Filter and write |

DuckDB reads the parquet out-of-core; 18.9M x 17 will not fit in pandas.

## Step 1 - Load

In [1]:
import json
from pathlib import Path

import duckdb
import pandas as pd

EXPORT = Path("/opt/data/export")
CLEANED = Path("/opt/data/ml/cleaned")

# hive_partitioning exposes start_year / start_month from the directory path
FACT = f"read_parquet('{EXPORT}/fact_trip/**/*.parquet', hive_partitioning=1)"
DIM_DATE = f"read_parquet('{EXPORT}/dim_date/*.parquet')"
DIM_STATION = f"read_parquet('{EXPORT}/dim_station/*.parquet')"

con = duckdb.connect()
# duckdb renders progress bars via ipywidgets, which this image does not carry
con.execute("SET enable_progress_bar_print=false")

print(f"duckdb {duckdb.__version__} | pandas {pd.__version__}")

duckdb 1.5.5 | pandas 2.3.3


In [2]:
# Reconcile against the manifest the export job wrote: if these disagree, the
# mount is stale or a partition is missing.
manifest = json.loads((EXPORT / "manifest.json").read_text())
expected = manifest["tables"]["fact_trip"]["rows"]

by_year = con.execute(f"""
    SELECT start_year, count(*) AS n_rows
    FROM {FACT}
    GROUP BY start_year ORDER BY start_year
""").df()

actual = by_year["n_rows"].sum()
print(by_year.to_string(index=False))
print(f"\ntotal {actual:,} | manifest {expected:,}")
assert actual == expected, "row count does not match the export manifest"

 start_year  n_rows
       2019 2439007
       2020 2908652
       2021 3569543
       2022 4295984
       2023 5707001

total 18,920,187 | manifest 18,920,187


In [3]:
con.execute(f"DESCRIBE SELECT * FROM {FACT}").df()[["column_name", "column_type"]]

,column_name,column_type
0,fact_trip_id,BIGINT
1,fact_trip_source_id,INTEGER
2,fact_trip_duration,INTEGER
3,fact_trip_start_ts,TIMESTAMP
4,fact_trip_end_ts,TIMESTAMP
5,fact_trip_start_date_id,DATE
6,fact_trip_end_date_id,DATE
7,fact_trip_start_hour,INTEGER
8,fact_trip_start_minute,INTEGER
9,fact_trip_end_hour,INTEGER


## Step 2 - Profile and validate

Cleaning thresholds come out of this step, not out of assumption.

In [4]:
# Nulls on the fields the target depends on, plus id uniqueness.
con.execute(f"""
    SELECT count(*) AS total,
        count(*) FILTER (fact_trip_start_station_id IS NULL) AS null_start_station,
        count(*) FILTER (fact_trip_start_ts IS NULL)         AS null_start_ts,
        count(*) FILTER (fact_trip_start_date_id IS NULL)    AS null_start_date,
        count(*) FILTER (fact_trip_start_hour IS NULL)       AS null_start_hour,
        count(*) FILTER (fact_trip_duration IS NULL)         AS null_duration,
        count(DISTINCT fact_trip_id)                         AS distinct_ids
    FROM {FACT}
""").df().T.rename(columns={0: "count"})

,count
total,18920187
null_start_station,0
null_start_ts,0
null_start_date,0
null_start_hour,0
null_duration,0
distinct_ids,18920187


In [5]:
# Referential integrity: every station the fact table departs from must exist
# in the dimension, or the Phase 2 grid will be built on unknown ids.
con.execute(f"""
    SELECT
        (SELECT count(*) FROM {DIM_STATION})                            AS dim_stations,
        (SELECT count(DISTINCT fact_trip_start_station_id) FROM {FACT}) AS active_stations,
        (SELECT count(*) FROM (SELECT DISTINCT fact_trip_start_station_id AS id FROM {FACT}) f
         LEFT JOIN {DIM_STATION} s ON f.id = s.dim_station_id
         WHERE s.dim_station_id IS NULL)                                AS orphan_stations,
        (SELECT count(*) FROM (SELECT DISTINCT fact_trip_start_date_id AS d FROM {FACT}) f
         LEFT JOIN {DIM_DATE} dd ON f.d = dd.dim_date_id
         WHERE dd.dim_date_id IS NULL)                                  AS orphan_dates
""").df().T.rename(columns={0: "count"})

,count
dim_stations,856
active_stations,852
orphan_stations,0
orphan_dates,0


In [6]:
# Calendar coverage. distinct_days < span means the system was idle on some
# days - those must be excluded from the Phase 2 grid, not zero-filled.
coverage = con.execute(f"""
    SELECT min(fact_trip_start_date_id) AS min_date,
        max(fact_trip_start_date_id)    AS max_date,
        count(DISTINCT fact_trip_start_date_id) AS distinct_days,
        count(DISTINCT fact_trip_start_hour)    AS distinct_hours
    FROM {FACT}
""").df()

span = (coverage["max_date"][0] - coverage["min_date"][0]).days + 1
print(coverage.T.rename(columns={0: "value"}).to_string())
print(f"\ncalendar span {span:,} days | missing {span - coverage['distinct_days'][0]:,}")

                              value
min_date        2019-01-01 00:00:00
max_date        2023-12-31 00:00:00
distinct_days                  1796
distinct_hours                   24

calendar span 1,826 days | missing 30


In [7]:
# Duration distribution sets the cleaning cutoffs.
con.execute(f"""
    SELECT min(fact_trip_duration) AS p_min,
        quantile_cont(fact_trip_duration, 0.01)  AS p01,
        quantile_cont(fact_trip_duration, 0.50)  AS p50,
        quantile_cont(fact_trip_duration, 0.99)  AS p99,
        quantile_cont(fact_trip_duration, 0.999) AS p999,
        max(fact_trip_duration) AS p_max,
        count(*) FILTER (fact_trip_duration < 60)    AS under_60s,
        count(*) FILTER (fact_trip_duration > 86400) AS over_24h
    FROM {FACT}
""").df().T.rename(columns={0: "seconds"})

,seconds
p_min,1.000000e+00
p01,1.100000e+02
p50,7.300000e+02
p99,5.382000e+03
p999,2.294281e+04
p_max,1.240378e+07
under_60s,4.095500e+04
over_24h,4.402000e+03


### Findings

Quality is high - no nulls on any field the target depends on, no duplicate
`fact_trip_id`, no orphan stations or dates, all 24 hours present.

Two things carry forward:

**Missing days.** 1,796 distinct days against an 1,826-day span - 30 days have
zero trips, most likely winter shutdowns. Phase 2 must exclude these from the
station-hour grid rather than zero-fill them, or the model learns "demand is 0"
for days the system was not running.

**Duration outliers.** Median is 730s (12 min), which is plausible. The tails
are not: 40,955 trips under 60s (false starts, re-docked immediately) and 4,402
over 24h (never returned, max ~143 days). Cutoff below drops ~0.24% of rows.

A sub-60s trip still represents someone taking a bike, so dropping it slightly
understates demand - but a 30-second round trip is not a rebalancing event.

In [8]:
MIN_DURATION = 60      # below this: false start, re-docked
MAX_DURATION = 86400   # above this: never returned

DURATION_FILTER = (
    f"fact_trip_duration >= {MIN_DURATION} "
    f"AND fact_trip_duration <= {MAX_DURATION}"
)

## Step 3 - Station pool

1. Top 50 by departures in at least one year.
2. Operating across the full 2019-2023 span.

Rule 2 drops 24 stations with a bounded lifespan - 18 opened late, 6 closed
early. Zero-filling those in Phase 2 would teach the model a station had no
demand when it had no dock.

In [9]:
TOP_N = 50
SPAN_START = "2019-01-31"  # first trip must be on or before
SPAN_END = "2023-12-01"    # last trip must be on or after

pool_df = con.execute(f"""
    WITH clean AS (
        SELECT fact_trip_start_station_id AS station_id, start_year,
            fact_trip_start_date_id AS d
        FROM {FACT}
        WHERE {DURATION_FILTER}
    ),
    top50 AS (
        SELECT station_id, start_year,
            row_number() OVER (PARTITION BY start_year ORDER BY count(*) DESC) AS rnk
        FROM clean GROUP BY station_id, start_year
        QUALIFY rnk <= {TOP_N}
    ),
    life AS (
        SELECT station_id, min(d) AS first_trip, max(d) AS last_trip
        FROM clean GROUP BY station_id
    )
    SELECT l.station_id, s.dim_station_name AS station_name,
        l.first_trip, l.last_trip,
        count(DISTINCT t.start_year) AS top50_years,
        min(t.rnk) AS best_rank
    FROM life l
    JOIN top50 t ON t.station_id = l.station_id
    LEFT JOIN {DIM_STATION} s ON s.dim_station_id = l.station_id
    WHERE l.first_trip <= DATE '{SPAN_START}'
      AND l.last_trip  >= DATE '{SPAN_END}'
    GROUP BY 1, 2, 3, 4 ORDER BY best_rank
""").df()

pool = sorted(pool_df["station_id"].tolist())
pool_sql = ", ".join(str(s) for s in pool)

print(f"pool: {len(pool)} stations")
pool_df.head(10)

pool: 76 stations


,station_id,station_name,first_trip,last_trip,top50_years,best_rank
0,7076,York St / Queens Quay W,2019-01-01,2023-12-31,5,1
1,7006,Bay St / College St (East Side),2019-01-01,2023-12-31,5,1
2,7016,Bay St / Queens Quay W (Ferry Terminal),2019-01-01,2023-12-31,5,2
3,7171,Ontario Place Blvd / Lake Shore Blvd W (East),2019-01-01,2023-12-31,4,2
4,7242,Lake Shore Blvd W / Ontario Dr,2019-01-01,2023-12-31,4,2
5,7203,Bathurst St/Queens Quay(Billy Bishop Airport),2019-01-01,2023-12-31,4,3
6,7033,Union Station,2019-01-01,2023-12-31,4,3
7,7030,Bay St / Wellesley St W,2019-01-01,2023-12-31,5,4
8,7175,HTO Park (Queens Quay W),2019-01-02,2023-12-31,4,4
9,7038,Dundas St W / Yonge St,2019-01-01,2023-12-31,5,5


In [10]:
# Every pool station must span the window, or Phase 2 needs per-station bounds.
assert (pool_df["first_trip"].max() <= pd.Timestamp(SPAN_START)), "a station opened late"
assert (pool_df["last_trip"].min() >= pd.Timestamp(SPAN_END)), "a station closed early"

print(f"{len(pool)} stations, all spanning "
      f"{pool_df['first_trip'].min().date()} to {pool_df['last_trip'].max().date()}")

76 stations, all spanning 2019-01-01 to 2023-12-31


## Step 4 - Filter and write

Apply the duration cutoffs and the station pool, then write to `data/cleaned`
partitioned by year. Only the columns the target and Phase 2 need are carried
forward - end-of-trip, bike, and user-type fields are not used for departure
counts.

In [11]:
# Rows surviving each filter, so the write is auditable rather than a black box.
audit = con.execute(f"""
    SELECT count(*) AS raw,
        count(*) FILTER ({DURATION_FILTER}) AS after_duration,
        count(*) FILTER ({DURATION_FILTER}
            AND fact_trip_start_station_id IN ({pool_sql})) AS after_pool
    FROM {FACT}
""").df().iloc[0]

for label, n in audit.items():
    print(f"{label:15s} {n:>12,}  ({100.0 * n / audit['raw']:5.1f}%)")

raw               18,920,187  (100.0%)
after_duration    18,874,830  ( 99.8%)
after_pool         5,890,830  ( 31.1%)


In [12]:
CLEANED.mkdir(parents=True, exist_ok=True)

con.execute(f"""
    COPY (
        SELECT fact_trip_id            AS trip_id,
            fact_trip_start_station_id AS station_id,
            fact_trip_start_date_id    AS start_date,
            fact_trip_start_hour       AS hour,
            fact_trip_duration         AS duration,
            start_year                 AS year
        FROM {FACT}
        WHERE {DURATION_FILTER}
          AND fact_trip_start_station_id IN ({pool_sql})
    ) TO '{CLEANED}' (FORMAT parquet, PARTITION_BY year, OVERWRITE_OR_IGNORE 1)
""")

print(f"wrote -> {CLEANED}")

wrote -> /opt/data/ml/cleaned


In [13]:
# Read back what landed on disk and reconcile against the audit above.
CLEAN = f"read_parquet('{CLEANED}/**/*.parquet', hive_partitioning=1)"

out = con.execute(f"""
    SELECT year, count(*) AS n_rows,
        count(DISTINCT station_id) AS stations,
        min(start_date) AS min_date, max(start_date) AS max_date
    FROM {CLEAN} GROUP BY year ORDER BY year
""").df()

print(out.to_string(index=False))
print(f"\ntotal {out['n_rows'].sum():,} | expected {audit['after_pool']:,}")
assert out["n_rows"].sum() == audit["after_pool"], "written rows do not match the filter"
assert out["stations"].max() <= len(pool), "station outside the pool was written"

 year  n_rows  stations   min_date   max_date
 2019  919848        76 2019-01-01 2019-12-31
 2020  936201        76 2020-01-01 2020-12-31
 2021 1107431        76 2021-01-01 2021-12-31
 2022 1291833        76 2022-01-01 2022-12-31
 2023 1635517        76 2023-01-01 2023-12-31

total 5,890,830 | expected 5,890,830


In [14]:
# Phase 2 needs the id list to build the grid; the app needs names.
pool_path = CLEANED.parent / "station_pool.parquet"
pool_df[["station_id", "station_name", "first_trip", "last_trip"]] \
    .sort_values("station_id").to_parquet(pool_path, index=False)

print(f"{len(pool_df)} stations -> {pool_path.name}")

76 stations -> station_pool.parquet
